In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/food_delivery_clean.csv")

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607_x,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,11:30,11:45,4,0,2,3,3,0.0,1,3,24.0
1,0xb379_x,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,19:45,19:50,3,1,2,3,4,1.0,1,1,33.0
2,0x20f7_x,BANGRES18DEL01,37,4.4,12.913041,77.683237,12.953041,77.723237,13:50,13:55,6,0,0,0,3,2.0,1,1,42.0
3,0x7804_x,HYDRES13DEL02,28,4.9,17.431477,78.400350,17.451477,78.420350,10:60,11:15,2,2,1,2,4,1.0,1,3,19.0
4,0x7faf_x,RANCHIRES07DEL01,37,4.6,23.359407,85.325055,23.429407,85.395055,23:50,23:60,3,2,0,2,3,1.0,1,1,25.0


Creating Delivery Distance 

In [3]:
#Currently, we only know the restaurant and customer coordinates.

#The model doesn't understand how far the driver travelled.

#Instead of giving it four latitude/longitude values, we compute the actual distance.

#We'll use the Haversine Formula, which calculates the shortest distance between two points on Earth.


from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2

    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c


df["Delivery_Distance_km"] = df.apply(
    lambda row: haversine(
        row["Restaurant_latitude"],
        row["Restaurant_longitude"],
        row["Delivery_location_latitude"],
        row["Delivery_location_longitude"]
    ),
    axis=1
)


df["Delivery_Distance_km"].describe()

count    2352.000000
mean       32.066580
std       349.749251
min         1.465159
25%         4.674292
50%         9.313161
75%        13.682075
max      6880.272782
Name: Delivery_Distance_km, dtype: float64

In [4]:


def clean_time(t):
    if pd.isna(t):
        return np.nan

    t = str(t).strip()

    hour, minute = map(int, t.split(":"))

    # Handle 24:xx
    if hour == 24:
        hour = 0

    # Handle xx:60
    if minute == 60:
        minute = 0
        hour += 1

        if hour == 24:
            hour = 0

    return f"{hour:02d}:{minute:02d}"

In [5]:
df["Time_Orderd"] = df["Time_Orderd"].apply(clean_time)
df["Time_Order_picked"] = df["Time_Order_picked"].apply(clean_time)

In [6]:
df["Time_Orderd"] = pd.to_datetime(df["Time_Orderd"], format="%H:%M")
df["Time_Order_picked"] = pd.to_datetime(df["Time_Order_picked"], format="%H:%M")

In [7]:
#Order Preparation Time

df["Preparation_Time"] = (
    df["Time_Order_picked"] - df["Time_Orderd"]
).dt.total_seconds() / 60

df.loc[df["Preparation_Time"] < 0, "Preparation_Time"] += 24 * 60

df["Order_Hour"] = df["Time_Orderd"].dt.hour
df["Pickup_Hour"] = df["Time_Order_picked"].dt.hour



In [8]:
print(df[["Preparation_Time", "Order_Hour", "Pickup_Hour"]].isnull().sum())



Preparation_Time    0
Order_Hour          0
Pickup_Hour         0
dtype: int64


In [9]:
print(df["Time_Orderd"].head())
print(type(df["Time_Orderd"].iloc[0]))

0   1900-01-01 11:30:00
1   1900-01-01 19:45:00
2   1900-01-01 13:50:00
3   1900-01-01 11:00:00
4   1900-01-01 23:50:00
Name: Time_Orderd, dtype: datetime64[us]
<class 'pandas.Timestamp'>


In [10]:
#Order Hour

df["Order_Hour"] = df["Time_Orderd"].dt.hour

In [11]:
#Pickup Hour

df["Pickup_Hour"] = df["Time_Order_picked"].dt.hour

In [12]:
#Pickup Hour Feature

df["Peak_Hour"] = np.where(
    ((df["Order_Hour"] >= 8) & (df["Order_Hour"] <=10)) |
    ((df["Order_Hour"] >=18) & (df["Order_Hour"] <=21)),
    1,
    0
)

df["Peak_Hour"].value_counts()

Peak_Hour
1    1280
0    1072
Name: count, dtype: int64

In [13]:
#Driver Experience Category

def rating_group(r):

    if r >= 4.8:
        return "Excellent"

    elif r >= 4.5:
        return "Good"

    else:
        return "Average"
    

df["Driver_Category"] = df["Delivery_person_Ratings"].apply(rating_group)    

In [14]:
#Driver Age Groups

df["Age_Group"] = pd.cut(

    df["Delivery_person_Age"],

    bins=[18,25,35,45,60],

    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-60"
    ]

)

In [15]:
#Distance Categories

df["Distance_Category"] = pd.cut(

    df["Delivery_Distance_km"],

    bins=[0,2,5,10,100],

    labels=[
        "Short",
        "Medium",
        "Long",
        "Very Long"
    ]

)

In [16]:
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Time_Orderd,Time_Order_picked,...,City,Time_taken(min),Delivery_Distance_km,Preparation_Time,Order_Hour,Pickup_Hour,Peak_Hour,Driver_Category,Age_Group,Distance_Category
0,0x4607_x,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,1900-01-01 11:30:00,1900-01-01 11:45:00,...,3,24.0,3.025149,15.0,11,11,0,Excellent,36-45,Medium
1,0xb379_x,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,1900-01-01 19:45:00,1900-01-01 19:50:00,...,1,33.0,20.183530,5.0,19,19,1,Good,26-35,Very Long
2,0x20f7_x,BANGRES18DEL01,37,4.4,12.913041,77.683237,12.953041,77.723237,1900-01-01 13:50:00,1900-01-01 13:55:00,...,1,42.0,6.210864,5.0,13,13,0,Average,36-45,Long
3,0x7804_x,HYDRES13DEL02,28,4.9,17.431477,78.400350,17.451477,78.420350,1900-01-01 11:00:00,1900-01-01 11:15:00,...,3,19.0,3.073618,15.0,11,11,0,Excellent,26-35,Medium
4,0x7faf_x,RANCHIRES07DEL01,37,4.6,23.359407,85.325055,23.429407,85.395055,1900-01-01 23:50:00,1900-01-01 00:00:00,...,1,25.0,10.564974,10.0,23,0,0,Good,36-45,Very Long


In [17]:
df.to_csv("../data/food_delivery_featured.csv", index=False)